In [23]:
# Import modules
import os
import sys

# Make sure that the executable path exists in the environment path
env_bin = os.path.dirname(sys.executable)
if env_bin not in os.environ["PATH"]:
    os.environ["PATH"] = f"{env_bin}:{os.environ['PATH']}"

import numpy as np
import pandas as pd
import xarray as xr
import yaml

with open("config.yml") as f:
    config = yaml.safe_load(f)
viz_opts = config["viz"]

In [24]:
from dask.distributed import Client, LocalCluster

cluster = LocalCluster(n_workers=4, threads_per_worker=4)
client = Client(cluster)
client

/albedo/soft/sw/conda-sw/analysis-toolbox-python-04.2026/lib/python3.12/site-packages/distributed/node.py:188: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 46243 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:46243/status,
Dashboard: http://127.0.0.1:46243/status,Workers: 4
Total threads: 16,Total memory: 32.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:35075,Workers: 0
Dashboard: http://127.0.0.1:46243/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:34433,Total threads: 4
Dashboard: http://127.0.0.1:36807/status,Memory: 8.00 GiB
Nanny: tcp://127.0.0.1:43975,


# Load ERA5 Data

In [25]:
# wind = xr.open_dataset("./data/era5_winddata.grib", engine="cfgrib", chunks='auto')
wind = xr.open_dataset(
    "./data/era5_2017_2020_winddata.grib",
    engine="cfgrib",
    chunks={"time": 5000, "latitude": 60, "longitude": 60},
)

In [26]:
wind

<xarray.Dataset> Size: 4GB
Dimensions:        (time: 13140, latitude: 161, longitude: 221)
Coordinates:
  * time           (time) datetime64[ns] 105kB 2017-01-01 ... 2019-12-31T22:0...
    valid_time     (time) datetime64[ns] 105kB dask.array<chunksize=(5000,), meta=np.ndarray>
  * latitude       (latitude) float64 1kB -50.0 -50.25 -50.5 ... -89.75 -90.0
  * longitude      (longitude) float64 2kB -75.0 -74.75 -74.5 ... -20.25 -20.0
    number         int64 8B ...
    step           timedelta64[ns] 8B ...
    isobaricInhPa  float64 8B ...
Data variables:
    u              (time, latitude, longitude) float32 2GB dask.array<chunksize=(5000, 60, 60), meta=np.ndarray>
    v              (time, latitude, longitude) float32 2GB dask.array<chunksize=(5000, 60, 60), meta=np.ndarray>
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-10-17T01:04 GRIB to CDM+CF via cfgrib-0.9.1...

In [27]:
# wind.compute()

In [28]:
coarsened_wind = wind.coarsen(latitude=2, longitude=2, boundary="trim").mean()
# coordinates are now block midpoints (x.25 -> x.125 etc.); snap to the half-degree grid
# introduces a 1/8 degree shift, but regridder crashes the kernel
coarsened_wind = coarsened_wind.assign_coords(
    latitude=np.round(coarsened_wind.latitude * 2) / 2,
    longitude=np.round(coarsened_wind.longitude * 2) / 2,
)

In [29]:
# coarsened_wind = xr.Dataset(
#     {
#         "latitude": (["latitude"], np.arange(-90, -49.9, 0.5)),
#         "longitude": (["longitude"], np.arange(-75, -19.5, 0.5)),
#     }
# )

# regridder = xe.Regridder(wind, coarsened_wind, "conservative_normalized")
# regridder

In [30]:
# the entire dataset can be processed at once
# coarsened_wind = regridder(ds)

In [31]:
coarsened_wind

<xarray.Dataset> Size: 925MB
Dimensions:        (time: 13140, latitude: 80, longitude: 110)
Coordinates:
  * time           (time) datetime64[ns] 105kB 2017-01-01 ... 2019-12-31T22:0...
    valid_time     (time) datetime64[ns] 105kB dask.array<chunksize=(5000,), meta=np.ndarray>
  * latitude       (latitude) float64 640B -50.0 -50.5 -51.0 ... -89.0 -89.5
  * longitude      (longitude) float64 880B -75.0 -74.5 -74.0 ... -21.0 -20.5
    number         int64 8B ...
    step           timedelta64[ns] 8B ...
    isobaricInhPa  float64 8B ...
Data variables:
    u              (time, latitude, longitude) float32 463MB dask.array<chunksize=(5000, 30, 30), meta=np.ndarray>
    v              (time, latitude, longitude) float32 463MB dask.array<chunksize=(5000, 30, 30), meta=np.ndarray>
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-10-17T01:04 GRIB to CDM+CF via cfgrib-0.9.1...

In [32]:
coarsened_wind = coarsened_wind.resample(
    time="120h", origin=pd.Timestamp("2017-02-12")
).mean()  # equivalent to 5 days

In [33]:
coarsened_wind

<xarray.Dataset> Size: 15MB
Dimensions:        (time: 220, latitude: 80, longitude: 110)
Coordinates:
  * time           (time) datetime64[ns] 2kB 2016-12-29 ... 2019-12-29
  * latitude       (latitude) float64 640B -50.0 -50.5 -51.0 ... -89.0 -89.5
  * longitude      (longitude) float64 880B -75.0 -74.5 -74.0 ... -21.0 -20.5
    number         int64 8B ...
    step           timedelta64[ns] 8B ...
    isobaricInhPa  float64 8B ...
Data variables:
    u              (time, latitude, longitude) float32 8MB dask.array<chunksize=(1, 30, 30), meta=np.ndarray>
    v              (time, latitude, longitude) float32 8MB dask.array<chunksize=(1, 30, 30), meta=np.ndarray>
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-10-17T01:04 GRIB to CDM+CF via cfgrib-0.9.1...

In [34]:
coarsened_wind.compute().to_zarr("surface_stress_wind.zarr", mode="w")

/albedo/soft/sw/conda-sw/analysis-toolbox-python-04.2026/lib/python3.12/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(
